In [35]:
# All imports in one cell
# Note: `pip install` commands go in Terminal

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All imports successful.')

All imports successful.


In [44]:
# Load the Dune API key from the '.env' file.
# Never paste the key directly into the notebook.
# That would expose it if the notebook is committed to GitHub.

from dotenv import load_dotenv
import os

load_dotenv()
DUNE_API_KEY = os.getenv('DUNE_API_KEY')

# Verify the key loaded correctly
# this should print s long dtring, not Noneprintable(repr(DUNE_API_KEY))

print(repr(DUNE_API_KEY))

'0cdwzDVCwupQW4pnMSemLVqvgsg2r9Uu'


In [46]:
from dune_client.client import DuneClient
from dune_client.query import QueryBase

client = DuneClient(api_key=DUNE_API_KEY)

# get_latest_result_dataframe() fetches cached results - no credit consumption
# Do not use run_query_dataframe() - triggers fresh execution, returns 400 on free plan

resuts = client.get_latest_result_dataframe(7365284)

print(results.head())
print(results.dtypes)

# Expected output:
# 936 rows covering October 2023 through April 2026
# Columns: date (object), transfer_count (int64), usdc_volume (float64)
# date is object at this stage, will be parsed in the next cell

                          date  transfer_count   usdc_volume
0  2023-10-01 00:00:00.000 UTC           42852  2.698518e+09
1  2023-10-02 00:00:00.000 UTC           61516  7.903753e+09
2  2023-10-03 00:00:00.000 UTC           50597  6.157389e+09
3  2023-10-04 00:00:00.000 UTC           56845  5.877146e+09
4  2023-10-05 00:00:00.000 UTC           54382  5.200415e+09
date               object
transfer_count      int64
usdc_volume       float64
dtype: object


In [47]:
# Save to CSV
# Save the API results immidiately to a local CSV. 
# This means API is only called once and all subsequent work reads from the saved file.
# this avoids hitting rate limits and makes the notebook reproducible withour an API connection.

resuts.to_csv('usdc_daily_volumes.csv', index=False)
print('Saved')

Saved


In [51]:
# Load and Parse USDC Data
# Load from the saved CSV. 'parse_dates='['date']' converts the date column from string to
# 'datetime64' automatically. 'dt.tz_localize(None)' removes the UTC timezone offset, 
# necessary for merging with the price data which has no timezone.

df = pd.read_csv('usdc_daily_volumes.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'USDC data: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Date range: {df.date.min()} to {df.date.max()}')
print(df.dtypes)
print(df.head())

# Expected output:
# 936 rows, 2.5 years of daily data, Oct 2023 to Apr 2026
# date: datetime64[ns, UTC], correctly parsed
# transfer_count: int64
# USDC_volume: float64 (in USD, already divided by 1e6 in the dune query)

USDC data: 936 rows, 3 columns
Date range: 2023-10-01 00:00:00+00:00 to 2026-04-23 00:00:00+00:00
date              datetime64[ns, UTC]
transfer_count                  int64
usdc_volume                   float64
dtype: object
                       date  transfer_count   usdc_volume
0 2023-10-01 00:00:00+00:00           42852  2.698518e+09
1 2023-10-02 00:00:00+00:00           61516  7.903753e+09
2 2023-10-03 00:00:00+00:00           50597  6.157389e+09
3 2023-10-04 00:00:00+00:00           56845  5.877146e+09
4 2023-10-05 00:00:00+00:00           54382  5.200415e+09


In [52]:
# Load BTC and ETH Price Data from SQLite
# The price data lived in the SQLite database from the companion 
# crypto-geopolitical-analysis project. We connect to it directly at its original path, 
# no need to copy it.

import sqlite3
conn = sqlite3.connect('/Users/sakicansev/Documents/Crypto-geopolitical-analysis/crypto_geopolitical.db')

# Verify available tables
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('Tables:', tables)
# Expected: [('crypto_prices',), ('geopolitical_events',)]

prices = pd.read_sql(
    'SELECT Date, BTC_Close, ETH_Close FROM crypto_prices ORDER BY Date',
    conn,
    parse_dates=['Date']
)
prices['date'] = prices['Date'].dt.normalize()
conn.close()

print(f'Price data: {prices.shape[0]} days')
print(prices.head())

# Expected output:
# 917 rows, covers same period as USDC data
# columnsL Date, BTC_Close, ETH_Close, date
# BTC starts around $27,983 on 2023-10-01, preconflict baseline

Tables: [('crypto_prices',), ('geopolitical_events',)]
Price data: 917 days
        Date     BTC_Close    ETH_Close       date
0 2023-10-01  27983.750000  1733.810425 2023-10-01
1 2023-10-02  27530.785156  1663.627563 2023-10-02
2 2023-10-03  27429.978516  1656.685669 2023-10-03
3 2023-10-04  27799.394531  1647.838135 2023-10-04
4 2023-10-05  27415.912109  1611.476440 2023-10-05


In [56]:
# Merge Datasets
# Merge USDC volume and price data on the 'date' column
# **Important:** The USDC `date` column has UTC timezone (`datetime64[ns, UTC]`) 
# while the price `date` has no timezone (`datetime64[ns]`). 
# Pandas will raise a `ValueError` if you try to merge mismatched timezones. 
# Fix: strip the timezone from the USDC column with `dt.tz_localize(None)` before merging.

# Strip UTC timezone from USDC date before merging
df['date'] = df['date'].dt.tz_localize(None)

merge = pd.merge(
    df,
    prices[['date', 'BTC_Close', 'ETH_Close']],
    on='date',
    how='inner'
)

print(f'Merged dataset: {merged.shape[0]} rows, {merged.shape[1]} columns')
print(f'Columns: {merged.columns.tolist()}')
print(f'Null values=\n{merged.isnull().sum()}')
print(merged.head())

# Expected output:
# 917 rows, 5 columns: date, transfer_count, usdc_volume, BTC_Close, ETH_Close
# Zero nll values - clean merge
# This is the primary analysis dataset for the entire event study

Merged dataset: 917 rows, 5 columns
Columns: ['date', 'transfer_count', 'usdc_volume', 'BTC_Close', 'ETH_Close']
Null values=
date              0
transfer_count    0
usdc_volume       0
BTC_Close         0
ETH_Close         0
dtype: int64
        date  transfer_count   usdc_volume     BTC_Close    ETH_Close
0 2023-10-01           42852  2.698518e+09  27983.750000  1733.810425
1 2023-10-02           61516  7.903753e+09  27530.785156  1663.627563
2 2023-10-03           50597  6.157389e+09  27429.978516  1656.685669
3 2023-10-04           56845  5.877146e+09  27799.394531  1647.838135
4 2023-10-05           54382  5.200415e+09  27415.912109  1611.476440


In [59]:
# Verify October 7 Data Point
# Before building the event study, verify the key finding from Stage 2 query 10 
# is present in the merged dataset. the October 7, 2023 Hamas attack should show 
# a sharp drop in USDC volume compared to the previous day.

# Check the three days around the October 7 event
oct_window = merged[(merged['date'] >= '2023-10-05') & (merged['date'] <= '2023-10-09')]
print(oct_window[['date', 'transfer_count', 'usdc_volume', 'BTC_Close', 'ETH_Close']].to_string())

# Expected output:
# Oct 6: ~54,835 transfer, ~$6.67B volume
# Oct 7: ~38,209 transfer, ~$2.24B volume <-- %66 drop
# Oct 8: ~37,698 transfer, ~$2.47B volume

# The drop is immidiately visible, markets froze on the day of the attack.
# This confirmed in python the same finding first identified in the Dune browser query.

        date  transfer_count   usdc_volume     BTC_Close    ETH_Close
4 2023-10-05           54382  5.200415e+09  27415.912109  1611.476440
5 2023-10-06           54835  6.667434e+09  27946.597656  1645.831543
6 2023-10-07           38209  2.244548e+09  27968.839844  1634.511230
7 2023-10-08           37698  2.474284e+09  27935.089844  1633.548584
8 2023-10-09           53344  5.347434e+09  27583.677734  1579.806641


In [60]:
# Save Merged Dataset

# Save the merged dataset locally so it can be loaded directly in future sessions without 
# running again the API call and merge.

merge.to_csv('merged_dataset.csv', index=False)
print('Saved')

Saved
